# SEC FSN Data Pipeline & Fundamentals Demo
* Section I: Initial Data Review & Exploration
* Section II: Sample Functions
* Section III: Sample pyxll examples
* Section IV: Sample plotly examples

## Section I: Initial Data Review & Exploration
* Download Data
* How to inspect & validate FSN data
* Run automated monitoring checks
* Validate schema and file presence
* How to build fundamentals using Polars
* Generate revenue, net income, margins, ROE, etc.
* How to display fundamentals for analysis

In [ ]:
import os
# os.chdir(r"ENTER NOTEBOOK FOLDER PATH")

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd()
print("CWD:", cwd)

# Determine the repo_root (folder containing the secfsn/ package)
if cwd.name == "Notebooks":
    # Notebook running inside .../secfsn/Notebooks
    repo_root = cwd.parent               # → .../secfsn
elif cwd.name == "secfsn":
    # Notebook running directly under the secfsn folder
    repo_root = cwd
else:
    # Fallback: search upward until we find a folder containing "secfsn"
    possible = [cwd] + list(cwd.parents)
    repo_root = None
    for p in possible:
        if (p / "secfsn").exists():
            repo_root = p
            break
    if repo_root is None:
        raise RuntimeError("❌ Could not locate secfsn/ package folder automatically.")

print("Detected REPO_ROOT:", repo_root)

# Add repo_root to sys.path (this makes import secfsn possible)
sys.path.insert(0, str(repo_root.parent))
print("Added to sys.path:", repo_root.parent)

# Test import
try:
    import secfsn
    print("✔ secfsn package imported successfully from:", secfsn.__file__)
except Exception as e:
    print("❌ secfsn import failed:", e)

# Now safe to import internals
try:
    from secfsn.config.core import DATA_DIR
    from secfsn.monitoring.run_checks import run_all_checks
    from secfsn.engine.polars_engine import build_fundamentals_polars_pandas
    print("✔ All internal imports succeeded.")
except Exception as e:
    print("❌ Internal import failure:", e)

In [ ]:
from secfsn.fsn.pipeline import run_fsn_pipeline
from secfsn.config.core import DATA_DIR

# Define the periods we want
quarters = [(2020, 4)]      # 2020 Q4
months   = [(2024, 10), (2025, 3), (2025, 6), (2025,9)]     # 2024 October, 2025 March, 2025 June, 2025 Septeber: Q4'24-Q3 25

print("Running FSN Pipeline for:")
print("  Quarterly:", quarters)
print("  Monthly:  ", months)
print("  DATA_DIR:", DATA_DIR)

# Uncomment the below if you want to run the pipeline
# run_fsn_pipeline(
#     quarters=quarters,
#     months=months,
#     base_dir=DATA_DIR,
#     overwrite_source=False   
# )

In [ ]:
# List FSN Data Periods

print("DATA_DIR:", DATA_DIR)

period_dirs = [p for p in DATA_DIR.iterdir() if p.is_dir()]
period_dirs

In [ ]:
# Run all monitoring checks

monitor_results = run_all_checks(DATA_DIR)
monitor_results

In [ ]:
# Load 1 quarter of fundamentals dataset

df = build_fundamentals_polars_pandas(year=2025, quarter=1, base_dir=DATA_DIR)
df.head()

In [ ]:
# Data Summary

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

df.describe(include="all").T.head(20)

## Section II — Sample Functions  
* Multi-period fundamentals panel with Polars
* Adding TTM
* Compre Companies
* Screener Example
* Ranking Example

In [ ]:
# Multi-period fundamentals panel with Polars engine

from secfsn.engine.polars_engine import build_fundamentals_polars_multi_pandas
from secfsn.config.core import DATA_DIR

# These periods match the FSN pipeline run in Section I
periods = [
    (2020, 4),   # Q4 2020
    (2024, 4),   # Q4 2024  → comes from monthly 2024-10
    (2025, 1),   # Q1 2025  → comes from monthly 2025-03
    (2025, 2),   # Q2 2025  → comes from monthly 2025-06
    (2025, 3),   # Q3 2025  → comes from monthly 2025-09
]

print("Building multi-period fundamentals panel...")
print("Periods:", periods)
print("DATA_DIR:", DATA_DIR)

df_panel = build_fundamentals_polars_multi_pandas(
    periods=periods,
    base_dir=DATA_DIR
)

print("\nMulti-period fundamentals loaded!")
print("Shape:", df_panel.shape)
df_panel.head()


In [ ]:
# Subset (top companies by revenue)
df_panel_demo = (
    df_panel
    .assign(revenue_num=pd.to_numeric(df_panel["revenue"], errors="coerce"))
    .sort_values("revenue_num", ascending=False)
    .head(500)
)

df_panel_demo.head()

In [ ]:
# TTM - on rolling filings, not strictly quarter accounting

def ttm(df, cik, column):
    """
    Compute TTM (rolling 4-quarter sum) for one company (CIK) and one metric.
    """

    temp = (
        df[df["cik"] == cik]
        .sort_values(["dataset_year", "dataset_quarter"])
        .reset_index(drop=True)
    )

    if temp.empty:
        raise ValueError(f"No rows found for CIK {cik}")

    if column not in temp.columns:
        raise ValueError(f"Column {column!r} does not exist.")

    # Rolling 4-filing sum
    ttm_vals = temp[column].rolling(4, min_periods=1).sum()

    result = pd.DataFrame({
        "period": temp["dataset_year"].astype(str)
                  + " Q"
                  + temp["dataset_quarter"].astype(str),
        f"ttm_{column}": ttm_vals
    })

    return result


# Demo — auto-select a valid company

# Find a company with at least 4 filings
cik_counts = df_panel_demo["cik"].value_counts()
demo_cik = cik_counts[cik_counts >= 4].index[0]

print("Using demo CIK:", demo_cik)

# Compute TTM revenue
ttm_output = ttm(df_panel_demo, demo_cik, "revenue")
ttm_output

In [ ]:
# Compare Companies

def compare_companies(df, cik1, cik2, metrics=None):
    """
    Compare two companies on selected fundamentals for the most recent filing.
    """

    if metrics is None:
        metrics = [
            "revenue", "net_income", "gross_margin", "operating_margin",
            "net_margin", "roa", "roe", "fcf_approx"
        ]

    # Fiscal period ordering
    fp_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4, "FY": 5}

    # Helper to extract the latest row for a given CIK
    def latest_row(cik):
        temp = df[df["cik"] == cik].copy()
        if temp.empty:
            raise ValueError(f"CIK {cik} not found in fundamentals.")

        # Map fp to numeric sort order
        temp["fp_order"] = temp["fp"].map(fp_order).fillna(0)

        # Sort by fiscal year then fiscal period rank
        temp = temp.sort_values(["fy", "fp_order"], ascending=True)

        return temp.tail(1)

    row1 = latest_row(cik1)
    row2 = latest_row(cik2)

    data = {
        "Metric": metrics,
        f"CIK {cik1}": [row1[m].item() if m in row1.columns else None for m in metrics],
        f"CIK {cik2}": [row2[m].item() if m in row2.columns else None for m in metrics],
    }

    return pd.DataFrame(data)

# --- Demo ---

import pandas as pd

def format_number(val):
    """Pretty-format numbers: billions, millions, and percentages."""
    if pd.isna(val):
        return None
    
    # percentage (if between -1 and 1)
    if -1 < val < 1:
        return f"{val*100:.2f}%"
    
    # billions
    if abs(val) >= 1e9:
        return f"{val/1e9:.2f}B"
    
    # millions
    if abs(val) >= 1e6:
        return f"{val/1e6:.2f}M"
    
    # otherwise plain
    return f"{val:,.0f}"

# --- Generate comparison table ---
# Microsoft 
msft = 789019 

# Apple 
aapl = 320193 

metrics_to_compare = [ "revenue", "net_income", "gross_margin", "operating_margin", "net_margin", "roa", "roe", "fcf_approx" ]

comparison = compare_companies(df_panel_demo, aapl, msft, metrics_to_compare)

# Apply pretty formatting
clean_comparison = comparison.copy()
clean_comparison.iloc[:, 1:] = clean_comparison.iloc[:, 1:].applymap(format_number)

clean_comparison.style.set_properties(
    **{
        "text-align": "center",
        "font-size": "14px"
    }
).set_table_styles(
    [
        dict(selector="th", props=[("text-align", "center"), ("font-size", "14px")]),
        dict(selector="td", props=[("text-align", "center")])
    ]
)

In [ ]:
# Screener: Latest Filing per Company

def format_number(val):
    """Pretty-format numeric values only (safe for mixed dtypes)."""
    if pd.isna(val):
        return None

    if not isinstance(val, (int, float, np.number)):
        return val

    # Ratios / percentages
    if -1 < val < 1:
        return f"{val * 100:.2f}%"

    # Billions
    if abs(val) >= 1e9:
        return f"{val / 1e9:.2f}B"

    # Millions
    if abs(val) >= 1e6:
        return f"{val / 1e6:.2f}M"

    return f"{val:,.0f}"


def simple_value_screener_latest(df):
    """
    Lightweight value-oriented screener applied to the
    MOST RECENT filing per company.

    Filters:
        - revenue >= $50M
        - gross margin >= 15%
        - ROE >= 10%
        - positive free cash flow
        - debt-to-equity < 2.0
    """

    temp = df.copy()

    # Coerce numeric columns safely
    numeric_cols = [
        "revenue",
        "gross_margin",
        "roe",
        "debt_to_equity",
        "fcf_approx",
    ]

    for col in numeric_cols:
        if col in temp.columns:
            temp[col] = pd.to_numeric(temp[col], errors="coerce")

    # --------------------------------------------------------
    # Keep ONLY the most recent filing per company (CIK)
    # --------------------------------------------------------
    df_latest = (
        temp
        .sort_values(["cik", "fy", "fp", "filed"])
        .groupby("cik", as_index=False)
        .tail(1)
    )

    # --------------------------------------------------------
    # Apply screener filters
    # --------------------------------------------------------
    screened = df_latest[
        (df_latest["revenue"] >= 50_000_000) &
        (df_latest["gross_margin"] >= 0.15) &
        (df_latest["roe"] >= 0.10) &
        (df_latest["fcf_approx"] >= 0) &
        (df_latest["debt_to_equity"] < 2.0)
    ].sort_values("roe", ascending=False)

    return screened


# ============================================================
# Demo
# ============================================================

display_cols = [
    "name",
    "fy",
    "fp",
    "revenue",
    "gross_margin",
    "roe",
    "debt_to_equity",
    "fcf_approx",
]

screener_results = simple_value_screener_latest(df_panel_demo)

print(f"Rows returned: {len(screener_results)}")

formatted = screener_results[display_cols].copy()

for col in display_cols:
    if col not in ["name", "fp"]:
        formatted[col] = formatted[col].apply(format_number)

formatted.reset_index(drop=True).head(15)


In [ ]:
# Ranking

def rank_companies(df, metric, top_n=20, ascending=False):
    """
    Rank companies by any numeric fundamental or ratio.

    Example:
        rank_companies(df, "roe", top_n=15)
        rank_companies(df, "fcf_approx", top_n=20)
        rank_companies(df, "revenue", top_n=30, ascending=False)

    Parameters:
        df (DataFrame)
        metric (str): Column to rank by
        top_n (int)
        ascending (bool): False = highest first, True = lowest first

    Returns:
        DataFrame: Top-N rows sorted by the metric.
    """

    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not found.")

    temp = df.copy()
    temp[metric] = pd.to_numeric(temp[metric], errors="coerce")

    ranked = (
        temp
        .dropna(subset=[metric])
        .sort_values(metric, ascending=ascending)
        .head(top_n)
    )

    return ranked

# --- Demo: Rank companies by ROE ---

# Columns we want to display
rank_cols = ["name", "roe", "revenue", "net_income", "fcf_approx"]

# Run the ranking
ranked_roe = rank_companies(df_panel_demo, "roe", top_n=15)

# Extract subset
rank_display = ranked_roe[rank_cols].copy()

# Apply numeric formatting using existing format_number
for col in rank_cols:
    if col != "name":
        rank_display[col] = rank_display[col].apply(format_number)

# Show nicely
print("Top 15 Companies by ROE:\n")
rank_display.reset_index(drop=True)


## Section III: Sample pyxll examples
*make sure to update config file with module and/or path*

1. py_build_fundamentals(years, quarters):Loads fundamentals using Polars pipeline → returns a DataFrame to Excel.
2. py_screen_value(df): Applies screener to the dataframe.
3. py_rank(df, metric, top_n): Ranks companies based on any metric.
4. General Formatting: Transforms raw dataframe → Excel-friendly with formatting (multiple options)

In [ ]:
from pyxll import xl_func, xl_return, Formatter, DataFrameFormatter
import pandas as pd
import numpy as np

from secfsn.engine.polars_engine import build_fundamentals_polars_multi_pandas
from secfsn.config.core import DATA_DIR

In [ ]:
@xl_func("object: dataframe<index=True>", auto_resize=True)
def expand_df(array):
    return array

@xl_func("object: dataframe<index=True>", auto_resize=True)
def expand_df_transpose(array):
    return array.transpose()

df_formatter = DataFrameFormatter(
    index=Formatter(bold=True, interior_color=Formatter.rgb(0xA9, 0xD0, 0x8E)),
    header=Formatter(bold=True, interior_color=Formatter.rgb(0xA9, 0xD0, 0x8E)),
    rows=[
        Formatter(interior_color=Formatter.rgb(0xE4, 0xF1, 0xDB)),
        Formatter(interior_color=Formatter.rgb(0xF4, 0xF9, 0xF1)),
    ]
)

@xl_func(formatter=df_formatter, auto_resize=True)
@xl_return("dataframe<index=True>")
def expand_df_formatted(array):
    return array

@xl_func(formatter=df_formatter, auto_resize=True)
@xl_return("dataframe<index=True>")
def expand_df_transpose_formatted(array):
    return array.transpose()

In [ ]:
def format_numeric_columns(input_df):
    def format_numeric(value):
        if pd.notnull(value):
            if abs(value) >= 1e9:
                return f"{value/1e9:.1f}B"
            elif abs(value) >= 1e6:
                return f"{value/1e6:.1f}M"
            elif abs(value) >= 1e3:
                return f"{value/1e3:.1f}k"
            else:
                return f"{value:,.2f}"
        return value

    formatted_df = input_df.copy()
    numeric_cols = formatted_df.select_dtypes(include=[np.number]).columns
    formatted_df[numeric_cols] = formatted_df[numeric_cols].applymap(format_numeric)
    return formatted_df

In [ ]:
@xl_func("int[] years, int[] quarters: object")
def py_build_fundamentals(years, quarters):
    """
    Return a DataFrame OBJECT HANDLE with a clean,
    Excel-friendly subset of fundamentals.

    Excel must use expand_df(...) to view it.
    """
    periods = list(zip(years, quarters))
    df = build_fundamentals_polars_multi_pandas(periods, DATA_DIR)

    # Prevent PyXLL cache collisions
    df["adsh"] = df["adsh"].astype(str)

    # Columns to expose in Excel (demo-friendly)
    cols = [
        "name",
        "cik",
        "fy",
        "fp",
        "revenue",
        "net_income",
        "gross_margin",
        "net_margin",
        "operating_margin",
        "roe",
        "roa",
        "fcf_approx",
    ]

    # Keep only columns that actually exist (defensive)
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    # Ensure numeric columns are numeric
    numeric_cols = [
        "revenue",
        "net_income",
        "gross_margin",
        "net_margin",
        "operating_margin",
        "roe",
        "roa",
        "fcf_approx",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sort by revenue
    if "revenue" in df.columns:
        df = df.sort_values("revenue", ascending=False)

    return df.head(10) # filter for sample
    # return df # if wanting no filter

In [ ]:
@xl_func("object df: object")
def py_value_screener(df):
    df = df.copy()

    num_cols = ["revenue", "gross_margin", "net_margin", "roe", "roa","debt_to_equity", "fcf_approx"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    out = df[
        (df["revenue"] >= 50_000_000) &
        (df["gross_margin"] >= 0.15) &
        (df["roe"] >= 0.10) &
        (df["fcf_approx"] >= 0)
    ].sort_values("roe", ascending=False)

    return out

In [ ]:
@xl_func("object df, string metric, int top_n, bool ascending: object")
def py_rank(df, metric, top_n=20, ascending=False):

    if metric not in df.columns:
        raise ValueError(f"{metric} not found.")

    df2 = df.copy()
    df2[metric] = pd.to_numeric(df2[metric], errors="coerce")

    out = (
        df2.dropna(subset=[metric])
           .sort_values(metric, ascending=ascending)
           .head(top_n)
    )
    return out

In [ ]:
@xl_func("object df: object")
def py_pretty(df):
    return format_numeric_columns(df)

## Section IV: Sample plotly examples

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Optional: set a consistent theme
px.defaults.template = "plotly_white"

In [ ]:
df = py_build_fundamentals([2024, 2025], [4, 1])

df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df_clean = df.dropna(subset=["revenue"])

df_top = (
    df_clean
    .sort_values("revenue", ascending=False)
    .head(1000)
    .reset_index(drop=True)
)

In [ ]:
fig = px.density_heatmap(
    df_top,
    x="roe",
    y="roa",
    nbinsx=30,
    nbinsy=30,
    color_continuous_scale="Viridis",
    title="ROE vs ROA — Density Heatmap (Top 1K by Revenue)",
    labels={"roe": "Return on Equity", "roa": "Return on Assets"}
)

fig.update_layout(height=500, width=700)
fig.show()

In [ ]:
fig = px.scatter(
    df_top,
    x="gross_margin",
    y="net_margin",
    size="revenue",
    color="roe",
    hover_name="name",
    hover_data=["cik", "revenue"],
    size_max=50,
    title="Profitability Map — Margins, ROE, and Revenue Size (Top 1K)",
    color_continuous_scale="RdBu"
)

fig.update_layout(height=550, width=750)
fig.show()

In [ ]:
companies = df_top["name"].unique()

fig = px.scatter(
    df_top,
    x="revenue",
    y="net_income",
    color="roe",
    hover_name="name",
    title="Interactive Fundamentals Viewer (Top 1K)"
)

# Add dropdown to filter by company
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="All Companies",
                    method="update",
                    args=[{"visible": [True] * len(df_top)}]
                )
            ] + [
                dict(
                    label=company,
                    method="update",
                    args=[
                        {
                            "visible": [
                                name == company for name in df_top["name"]
                            ]
                        }
                    ]
                )
                for company in companies
            ],
            direction="down",
            showactive=True,
            x=1.15,
            xanchor="left",
            y=1.0,
            yanchor="top"
        )
    ]
)

fig.update_layout(height=600, width=850)
fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def build_pl_waterfall(row):
    revenue = pd.to_numeric(row.get("revenue", 0), errors="coerce") or 0
    net_income = pd.to_numeric(row.get("net_income", 0), errors="coerce") or 0

    operating_margin = pd.to_numeric(row.get("operating_margin", np.nan), errors="coerce")

    # Derive operating income if margin exists
    operating_income = (
        revenue * operating_margin
        if pd.notna(operating_margin)
        else None
    )

    x = ["Revenue"]
    y = [revenue]
    measure = ["absolute"]

    # Optional Gross Profit
    gross_margin = pd.to_numeric(row.get("gross_margin", np.nan), errors="coerce")
    if pd.notna(gross_margin):
        gross_profit = revenue * gross_margin
        x.append("Gross Profit")
        y.append(gross_profit - revenue)
        measure.append("relative")
        base_for_op = gross_profit
    else:
        base_for_op = revenue

    # Operating Income (derived, always included if margin exists)
    if pd.notna(operating_income):
        x.append("Operating Income")
        y.append(operating_income - base_for_op)
        measure.append("relative")
        base_for_net = operating_income
    else:
        base_for_net = base_for_op

    # Net Income (always included)
    x.append("Net Income")
    y.append(net_income - base_for_net)
    measure.append("relative")

    return x, y, measure


# ------------------------------------------------------------
# Prepare companies
# ------------------------------------------------------------
companies = df_top["name"].unique()

default_company = companies[0]
default_row = df_top[df_top["name"] == default_company].iloc[0]

x0, y0, m0 = build_pl_waterfall(default_row)

# ------------------------------------------------------------
# Initial figure
# ------------------------------------------------------------
fig = go.Figure(
    go.Waterfall(
        name=default_company,
        orientation="v",
        measure=m0,
        x=x0,
        y=y0,
        textposition="outside",
        increasing={"marker": {"color": "#2ecc71"}},
        decreasing={"marker": {"color": "#e74c3c"}},
        totals={"marker": {"color": "#34495e"}},
    )
)

# ------------------------------------------------------------
# Dropdown
# ------------------------------------------------------------
buttons = []

for company in companies:
    row = df_top[df_top["name"] == company].iloc[0]
    x, y, m = build_pl_waterfall(row)

    buttons.append(
        dict(
            label=company,
            method="update",
            args=[
                {"x": [x], "y": [y], "measure": [m]},
                {"title": f"P&L Bridge – {company}"}
            ],
        )
    )

fig.update_layout(
    title=f"P&L Bridge – {default_company}",
    height=520,
    width=760,
    yaxis_title="USD",
    waterfallgap=0.3,
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.02,
            xanchor="left",
            y=1.0,
            yanchor="top",
        )
    ],
)

fig.show()